In [1]:
import os
import io
import webbrowser
from contextlib import redirect_stdout
from typing import Dict, Tuple, Optional, Callable, Any
import numpy as np
import pinocchio as pin
from pinocchio.utils import *
from pinocchio.visualize import MeshcatVisualizer
import tkinter as tk
from tkinter import filedialog
import ipywidgets as widgets
from IPython.display import display, HTML

urdf_filepath = "exo1.urdf"
package_dirs= [os.getcwd()]

Fußlinks = "left_ankle_fe"
Fußrechts = "right_ankle_fe"


In [2]:
button_style = {'description_width': 'initial', 'button_color': '#f0f0f0'}
button_layout = widgets.Layout(width='auto', margin='5px')


In [3]:
neutral_angles: Dict[str, float] = {
    "left_hip_aa": -0.00,
    "left_hip_fe": -10.00,
    "left_knee_fe": -7.00,
    Fußlinks: -7.00,
    "right_hip_aa": 0.00,
    "right_hip_fe": 10.00,
    "right_knee_fe": 7.00,
    Fußrechts: 7.00,
}

joint_limits_rad: Dict[str, Tuple[float, float]] = {
    "left_hip_aa": (np.deg2rad(-6), np.deg2rad(15)),  
    "left_hip_fe": (np.deg2rad(-16), np.deg2rad(121)),  
    "left_knee_fe": (np.deg2rad(-125), np.deg2rad(-7)),               
    "right_hip_aa": (np.deg2rad(-15), np.deg2rad(3)),  
    "right_hip_fe": (np.deg2rad(-16), np.deg2rad(121)),
    "right_knee_fe": (np.deg2rad(-125), np.deg2rad(-7)),
}


points = []  # Speichert die Punktkoordinaten
durations = [] # Speichert die Dauer zum Erreichen der Punkte



In [4]:
# URDF-Modell laden
model = pin.buildModelFromUrdf(urdf_filepath)
data = model.createData()
q_neutral = pin.neutral(model)

current_q = q_neutral.copy()


def update_neutral_position() -> None:
    global q_neutral
    q_neutral = pin.neutral(model)
    
    for joint_name, angle in neutral_angles.items():
        joint_id = model.getJointId(joint_name)
        if joint_id <= model.njoints:
            q_idx = model.idx_qs[joint_id]
            q_neutral[q_idx] = np.deg2rad(angle)
        else:
            raise KeyError(f"Gelenk '{joint_name}' nicht im Modell gefunden.")
    
    viz.display(q_neutral)

# Visualisierungsmodelle erstellen
visual_model = pin.buildGeomFromUrdf(model, urdf_filepath, pin.GeometryType.VISUAL, package_dirs=package_dirs)
collision_model = pin.buildGeomFromUrdf(model, urdf_filepath, pin.GeometryType.COLLISION,  package_dirs=package_dirs)

def suppress_terminal_output() -> io.StringIO:
    return redirect_stdout(io.StringIO())

# Initialisierung des MeshcatVisualizers mit unterdrückter Ausgabe
if 'viz' not in globals():
    with suppress_terminal_output():
        viz = MeshcatVisualizer(model, collision_model, visual_model)
        viz.initViewer(loadModel=True)
        viz.loadViewerModel()
        viz.display(q_neutral)


In [5]:
# Laden von Externen Splines zur Vizualisierung

def load_trajectorie_txt(path: str) -> np.ndarray:
    try:
        
        traj = np.loadtxt(path, delimiter=',', skiprows=1)
        traj_werte = traj[:, 1:]
       
        traj_joints = traj_werte
        """
        traj_joints = np.zeros_like(traj_werte)
        traj_joints[:, 0:4] = traj_werte[:, 4:8] # R -> Ls
        traj_joints[:, 4:8] = traj_werte[:, 0:4] # L -> R
        """
        corrections = np.array([-1, -1, -1, -1, -1, 1, 1, 1])
        traj_joints = traj_joints * corrections

        start_idx = 0 if model.nq == 8 else 7
        traj_joints = traj_joints + q_neutral[start_idx:]

        
        print(f"Trajektorie erfolgreich geladen: {traj_joints.shape[0]} Wegpunkte.")
        return traj_joints
        
    except Exception as e:
        print(f"Fehler beim Laden der Datei: {e}")
        return None
        
import meshcat.animation as anim

def play_trajectory(traj: np.ndarray, dt: float = 0.05):
    if traj is None:
        print("Keine Trajektorie zum Abspielen vorhanden.")
        return

    fps = int(1/dt)
    animation = anim.Animation(default_framerate=fps)
    
    print(f"Animation wird für {len(traj)} Frames vorbereitet...")
    
    for i in range(len(traj)):
        q_small = traj[i]
        q_full = pin.neutral(model)
        
        if len(q_small) == 8:
            q_full[0:8] = q_small

        else:
            q_full = q_small
        # Wir sagen MeshCat: "Speichere diese Pose für Frame i"
        with animation.at_frame(viz.viewer, i) as frame:
            viz.display(q_full)
            
    # Die Animation an den Viewer übertragen
    # play=True lässt sie sofort einmal loslaufen
    viz.viewer.set_animation(animation, play=True) 
    
    print("--------------------------------------------------")
    print("Fertig! Die Animation wurde an den Browser gesendet.")
    print("Du kannst sie jetzt im MeshCat-Tab steuern/aufnehmen.")
    print("--------------------------------------------------")
    

def open_meshcat_viewer(_) -> None:

    viewer_url = viz.viewer.url()
    webbrowser.open(viewer_url, new=2)  

def display_model(q: np.ndarray) -> None:
    if len(q) != model.nq:
        raise ValueError(f"Die Konfiguration muss {model.nq} Werte enthalten.")
    
    viz.display(q)

def update_joint_position(joint_name: str, x: float, y: float, z: float, tol: Optional[float] = 1e-5) -> Optional[np.ndarray]:

    global current_q
    global joint_limits_rad
    frame_id = model.getFrameId(joint_name)
    target_position = np.array([x, y, z])
    display_target(target_position, name=joint_name)
    try:
        new_q, success, used_bisection = ik_solver.solve_linesearch(side=joint_name, target_position=target_position, q0=current_q) 
        print(f"DEBUG in update_joint_position: success? {success}  -  bisection used? {used_bisection}  -  q={new_q}")
        viz.display(new_q)
        if success:
            current_q[:] = new_q
        return new_q
    except RuntimeError as e:
        print(f"Fehler bei der Inversen Kinematik: {e}")



In [6]:
here = os.getcwd()  
urdfpath = os.path.join(here, "exo1.urdf")


def on_load_traj_clicked(_):
    global loaded_trajectory
    
    # 1. Dateidialog öffnen
    root = tk.Tk()
    root.withdraw()
    # Stellt sicher, dass das Fenster vor dem Browser erscheint
    root.attributes("-topmost", True) 
    file_path = filedialog.askopenfilename(
        title="MATLAB Trajektorie auswählen",
        filetypes=[("Text files", "*.txt"), ("All files", "*.*")]
    )
    root.destroy()

    # 2. Wenn eine Datei ausgewählt wurde, laden
    if file_path:
        print(f"Lade externe Trajektorie: {file_path}")
        # Deine Logik-Funktion aufrufen
        loaded_trajectory = load_trajectorie_txt(file_path)
        
        if loaded_trajectory is not None:
            print(f"Erfolg: {len(loaded_trajectory)} Zeitschritte geladen.")
    else:
        print("Laden abgebrochen.")

def on_play_traj_clicked(_):
    if loaded_trajectory is not None:
        # 0.02 entspricht 50Hz, pass das an dein MATLAB-dt an
        play_trajectory(loaded_trajectory, dt=0.02)
    else:
        print("Fehler: Zuerst eine Trajektorie laden!")




In [7]:
load_traj_button = widgets.Button(description="Trajektorie Laden", style=button_style, layout=button_layout)
load_traj_button.on_click(on_load_traj_clicked)

play_traj_button = widgets.Button(description="Trajektorie Abspielen", style=button_style, layout=button_layout)
play_traj_button.on_click(on_play_traj_clicked)

open_meshcat_button = widgets.Button(description="MeshCat öffnen", layout=button_layout ,style=button_style)
open_meshcat_button.on_click(open_meshcat_viewer)
    
Neutralposition_button = widgets.Button(description="Neutralposition anzeigen", layout=button_layout,style=button_style)
Neutralposition_button.on_click(lambda _: update_neutral_position())

buttons = widgets.HBox([open_meshcat_button,  load_traj_button, Neutralposition_button, play_traj_button])

# Hauptcontainer für das GUI
main_container = widgets.VBox([
    widgets.HTML(value="<h2>RISE Exoskelett Trajektorien-Visualisierer</h2>"),
    buttons,
])

# Anzeigen des Hauptcontainers
display(main_container)